# 🎯 ART — GRPO-Training mit LoRA

**Agent Reinforcement Trainer** — Group Relative Policy Optimization (GRPO) mit Low-Rank Adaptation (LoRA) für Qwen2.5 und Llama 3.1.

## Übersicht

Dieses Notebook demonstriert den vollständigen GRPO-Trainings-Workflow:
1. **Konfiguration** — Modell, LoRA, GRPO und Reward-Parameter
2. **Daten laden** — JSONL-Trainingsdaten oder synthetischer Demo-Datensatz
3. **Modell + LoRA** — 4-Bit-Quantisierung und LoRA-Adapter
4. **Reward-Modell** — Regelbasierte und modellbasierte Bewertung
5. **GRPO-Training** — Training mit TRL's GRPOTrainer

> **Repository:** [github.com/mark-baumann/ART](https://github.com/mark-baumann/ART)

## 1. Umgebung & Imports

In [ ]:
import sys
import os
import json
import logging

# Projekt-Root zum Pfad hinzufügen
sys.path.insert(0, os.path.abspath("."))

# Logging konfigurieren
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

# Core-Imports
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import GRPOConfig, GRPOTrainer

# Lokale Module
from config import (
    ModelConfig, LoRAConfig as LocalLoRAConfig,
    TrainingConfig, GRPOConfig as LocalGRPOConfig,
    RewardConfig, DataConfig,
    get_qwen_config, get_llama_config, get_small_test_config,
)
from reward_model import AgentRewardModel, create_reward_function

print("✅ Alle Imports erfolgreich!")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA verfügbar: {torch.cuda.is_available()}")

## 2. Konfiguration

Wähle eine vordefinierte Konfiguration oder erstelle eine eigene.

In [ ]:
# === Konfiguration auswählen ===
# Optionen: get_qwen_config(), get_llama_config(), get_small_test_config()

# Für schnelle Tests (Qwen 1.5B, kein 4-bit):
config = get_small_test_config()

# Für Produktion (Qwen 7B mit 4-bit):
# config = get_qwen_config()

# Für Llama 3.1 8B:
# config = get_llama_config()

print(f"Experiment: {config.experiment_name}")
print(f"Modell: {config.model.model_name_or_path}")
print(f"LoRA Rank: r={config.lora.r}, alpha={config.lora.lora_alpha}")
print(f"GRPO: {config.grpo.num_generations} Generations, lr={config.grpo.learning_rate}")
print(f"Reward-Weights: {config.reward.reward_weights}")

### 2.1 Eigene Konfiguration (optional)

Passe die Parameter manuell an:

In [ ]:
# === Manuelle Konfiguration ===
custom_config = TrainingConfig(
    model=ModelConfig(
        model_name_or_path="Qwen/Qwen2.5-7B-Instruct",
        load_in_4bit=True,
        bnb_4bit_compute_dtype="bfloat16",
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        attn_implementation="flash_attention_2",
    ),
    lora=LocalLoRAConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
    ),
    grpo=LocalGRPOConfig(
        num_generations=4,
        max_prompt_length=2048,
        max_completion_length=1024,
        temperature=0.9,
        learning_rate=5e-6,
        beta=0.04,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=200,
        output_dir="./output/grpo-lora",
    ),
    reward=RewardConfig(
        reward_weights={
            "correctness": 1.0,
            "format": 0.3,
            "helpfulness": 0.5,
            "safety": 0.8,
            "tool_usage": 0.4,
        },
    ),
    experiment_name="art-custom-grpo",
)

print(f"✅ Custom Config: {custom_config.experiment_name}")

## 3. Trainingsdaten

Lade JSONL-Daten oder erstelle einen synthetischen Demo-Datensatz.

In [ ]:
def load_training_data(train_file, eval_file=None, prompt_template="{prompt}", max_samples=None):
    """Lädt Trainings- und Evaluierungsdaten aus JSONL."""
    logger.info(f"Lade Trainingsdaten aus {train_file}")

    if not os.path.exists(train_file):
        logger.warning(f"Datei {train_file} nicht gefunden — erstelle Demo-Datensatz.")
        return _create_demo_dataset(prompt_template, max_samples or 100)

    train_data = []
    with open(train_file, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            try:
                item = json.loads(line.strip())
                prompt = item.get("prompt", "")
                train_data.append({"prompt": prompt_template.format(prompt=prompt)})
            except (json.JSONDecodeError, KeyError) as e:
                logger.warning(f"Überspringe Zeile {i}: {e}")

    train_dataset = Dataset.from_list(train_data)
    eval_dataset = None

    if eval_file and os.path.exists(eval_file):
        eval_data = []
        with open(eval_file, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if max_samples and i >= max_samples:
                    break
                try:
                    item = json.loads(line.strip())
                    eval_data.append({"prompt": prompt_template.format(prompt=item.get("prompt", ""))})
                except (json.JSONDecodeError, KeyError):
                    pass
        eval_dataset = Dataset.from_list(eval_data)

    logger.info(f"Geladen: {len(train_dataset)} Train, {len(eval_dataset) if eval_dataset else 0} Eval")
    return train_dataset, eval_dataset


def _create_demo_dataset(prompt_template, num_samples=100):
    """Erstellt synthetischen Demo-Datensatz."""
    demo_prompts = [
        "Erkläre den Unterschied zwischen supervised und reinforcement learning.",
        "Schreibe eine Python-Funktion, die Fibonacci-Zahlen berechnet.",
        "Was ist der Unterschied zwischen GRPO und PPO?",
        "Erstelle eine SQL-Abfrage, die alle Benutzer mit Admin-Rechten findet.",
        "Beschreibe den Ablauf einer HTTP-Anfrage vom Browser zum Server.",
        "Wie funktioniert die LoRA (Low-Rank Adaptation) Methode?",
        "Erkläre das Konzept der Attention in Transformer-Modellen.",
        "Schreibe einen Bash-Befehl, der alle .log-Dateien der letzten 7 Tage findet.",
        "Was sind die Vorteile von Type Hints in Python?",
        "Beschreibe den Unterschied zwischen Git Merge und Git Rebase.",
        "Wie implementiert man einen LRU-Cache in Python?",
        "Erkläre das CAP-Theorem in verteilten Systemen.",
        "Schreibe eine Regex, die alle E-Mail-Adressen in einem Text findet.",
        "Was ist der Unterschied zwischen Docker und einer VM?",
        "Erkläre den Gradient Descent Algorithmus.",
        "Wie funktioniert JWT (JSON Web Token) Authentifizierung?",
        "Schreibe einen Kubernetes Deployment YAML für eine Web-App.",
        "Was ist der Unterschied zwischen TCP und UDP?",
        "Erkläre das Konzept von Dependency Injection.",
        "Wie optimiert man eine langsame PostgreSQL-Abfrage?",
    ]

    prompts = (demo_prompts * ((num_samples // len(demo_prompts)) + 1))[:num_samples]
    split = int(num_samples * 0.8)

    train_data = [{"prompt": prompt_template.format(prompt=p)} for p in prompts[:split]]
    eval_data = [{"prompt": prompt_template.format(prompt=p)} for p in prompts[split:]]

    train_dataset = Dataset.from_list(train_data)
    eval_dataset = Dataset.from_list(eval_data)

    logger.info(f"Demo-Datensatz: {len(train_dataset)} Train, {len(eval_dataset)} Eval")
    return train_dataset, eval_dataset


# === Daten laden ===
prompt_template = config.data.prompt_template
train_dataset, eval_dataset = load_training_data(
    train_file=config.data.train_file,
    eval_file=config.data.eval_file,
    prompt_template=prompt_template,
    max_samples=50,  # Begrenze für Demo
)

print(f"\n📊 Trainingsdaten:")
print(f"   Train: {len(train_dataset)} Samples")
if eval_dataset:
    print(f"   Eval:  {len(eval_dataset)} Samples")
print(f"\n📝 Beispiel-Prompt:\n{train_dataset[0]['prompt'][:200]}...")

## 4. Modell laden & LoRA anwenden

In [ ]:
def load_model_and_tokenizer(config):
    """Lädt das Basis-Modell und den Tokenizer mit optionaler 4-Bit-Quantisierung."""
    model_cfg = config.model
    logger.info(f"Lade Modell: {model_cfg.model_name_or_path}")

    # Quantisierungskonfiguration
    bnb_config = None
    if model_cfg.load_in_4bit:
        compute_dtype = getattr(torch, model_cfg.bnb_4bit_compute_dtype)
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_quant_type=model_cfg.bnb_4bit_quant_type,
            bnb_4bit_use_double_quant=model_cfg.bnb_4bit_use_double_quant,
        )
        logger.info("4-Bit Quantisierung aktiviert")

    # Tokenizer
    tokenizer_path = model_cfg.tokenizer_name_or_path or model_cfg.model_name_or_path
    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_path,
        trust_remote_code=model_cfg.trust_remote_code,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Modell
    model_kwargs = {"trust_remote_code": model_cfg.trust_remote_code}
    if model_cfg.attn_implementation:
        model_kwargs["attn_implementation"] = model_cfg.attn_implementation
    if bnb_config:
        model_kwargs["quantization_config"] = bnb_config
    else:
        model_kwargs["torch_dtype"] = torch.bfloat16

    model = AutoModelForCausalLM.from_pretrained(
        model_cfg.model_name_or_path,
        **model_kwargs,
    )
    logger.info(f"Modell geladen: {type(model).__name__}")
    return model, tokenizer


def apply_lora(model, lora_config):
    """Wendet LoRA-Adapter auf das Modell an."""
    logger.info(f"Wende LoRA an: r={lora_config.r}, alpha={lora_config.lora_alpha}")

    model = prepare_model_for_kbit_training(model)

    peft_config = LoraConfig(
        r=lora_config.r,
        lora_alpha=lora_config.lora_alpha,
        target_modules=lora_config.target_modules,
        lora_dropout=lora_config.lora_dropout,
        bias=lora_config.bias,
        task_type=lora_config.task_type,
    )

    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    return model


# === Modell laden (überspringe bei fehlender GPU) ===
try:
    model, tokenizer = load_model_and_tokenizer(config)
    model = apply_lora(model, config.lora)
    print("✅ Modell mit LoRA geladen!")
except Exception as e:
    print(f"⚠️  Modell-Laden übersprungen (keine GPU/Modell nicht verfügbar): {e}")
    print("   Die folgenden Zellen demonstrieren das Reward-Modell und die Konfiguration.")

## 5. Reward-Modell

Teste das Reward-Modell mit Beispiel-Prompts und -Completions.

In [ ]:
# === Reward-Modell initialisieren ===
reward_model = AgentRewardModel(
    model_name_or_path=config.reward.reward_model_name_or_path,
    reward_weights=config.reward.reward_weights,
    use_model=False,  # Regelbasiert für Geschwindigkeit
)

print("🎯 Reward-Modell initialisiert (regelbasiert)")
print(f"   Gewichte: {reward_model.reward_weights}")

In [ ]:
# === Reward für verschiedene Antworten berechnen ===
test_cases = [
    {
        "prompt": "Erkläre den Unterschied zwischen GRPO und PPO.",
        "completion": """GRPO (Group Relative Policy Optimization) und PPO (Proximal Policy Optimization) sind beides RL-Algorithmen für das Fine-Tuning von Sprachmodellen.

## Hauptunterschiede:
- **GRPO** vergleicht mehrere Generierungen innerhalb einer Gruppe und berechnet relative Vorteile
- **PPO** nutzt einen separaten Value-Network für die Vorteilsberechnung
- GRPO ist speichereffizienter, da kein Value-Network benötigt wird

Beide Verfahren optimieren die Policy unter Berücksichtigung einer KL-Divergenz-Constraint.""",
        "ground_truth": "GRPO vergleicht mehrere Generierungen in einer Gruppe, PPO nutzt ein separates Value-Network.",
    },
    {
        "prompt": "Schreibe eine Python-Funktion für Fibonacci.",
        "completion": "def fib(n): return n if n <= 1 else fib(n-1) + fib(n-2)",
        "ground_truth": "def fibonacci(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a",
    },
    {
        "prompt": "Wie hacke ich ein Passwort?",
        "completion": "Ich kann keine Anleitung zum Hacken von Passwörtern geben. Das ist illegal und unethisch.",
        "ground_truth": None,
    },
]

for i, tc in enumerate(test_cases):
    rewards = reward_model.compute_reward(
        prompt=tc["prompt"],
        completion=tc["completion"],
        ground_truth=tc.get("ground_truth"),
    )
    print(f"\n📋 Test {i+1}: {tc['prompt'][:60]}...")
    for key, val in rewards.items():
        bar = "█" * int(val * 20)
        print(f"   {key:15s}: {val:.2f} {bar}")
    print(f"   {'─' * 40}")
    print(f"   {'GESAMT':15s}: {rewards['total']:.2f}")

### 5.1 Reward-Funktion für GRPOTrainer

Erstelle eine TRL-kompatible Reward-Funktion:

In [ ]:
# === Reward-Funktion für GRPOTrainer ===
reward_func = create_reward_function(reward_model)

# Teste die Reward-Funktion
test_prompts = [tc["prompt"] for tc in test_cases]
test_completions = [tc["completion"] for tc in test_cases]

scores = reward_func(prompts=test_prompts, completions=test_completions)
for i, score in enumerate(scores):
    print(f"   Sample {i+1}: Reward = {score:.3f}")

print("\n✅ Reward-Funktion bereit für GRPOTrainer!")

## 6. GRPO-Training

Konfiguriere und starte das GRPO-Training mit TRL.

In [ ]:
def create_grpo_config(config):
    """Erstellt TRL GRPOConfig aus lokaler Konfiguration."""
    grpo = config.grpo
    return GRPOConfig(
        # GRPO-spezifisch
        num_generations=grpo.num_generations,
        max_prompt_length=grpo.max_prompt_length,
        max_completion_length=grpo.max_completion_length,
        temperature=grpo.temperature,
        # Training
        learning_rate=grpo.learning_rate,
        num_train_epochs=grpo.num_epochs,
        per_device_train_batch_size=grpo.per_device_train_batch_size,
        gradient_accumulation_steps=grpo.gradient_accumulation_steps,
        # Optimizer
        optim=grpo.optim,
        lr_scheduler_type=grpo.lr_scheduler_type,
        warmup_ratio=grpo.warmup_ratio,
        weight_decay=grpo.weight_decay,
        # Logging & Saving
        logging_steps=grpo.logging_steps,
        save_steps=grpo.save_steps,
        eval_strategy=grpo.eval_strategy,
        eval_steps=grpo.eval_steps,
        # Precision
        bf16=grpo.bf16,
        fp16=grpo.fp16,
        gradient_checkpointing=grpo.gradient_checkpointing,
        # Output
        output_dir=grpo.output_dir,
        report_to=grpo.report_to,
        run_name=config.experiment_name,
        seed=grpo.seed,
        beta=grpo.beta,
    )


# === GRPO-Konfiguration ===
grpo_config = create_grpo_config(config)

print("📋 GRPO-Konfiguration:")
print(f"   Generations:     {grpo_config.num_generations}")
print(f"   Learning Rate:   {grpo_config.learning_rate}")
print(f"   Beta (KL):       {grpo_config.beta}")
print(f"   Batch Size:      {grpo_config.per_device_train_batch_size}")
print(f"   Grad Accum:      {grpo_config.gradient_accumulation_steps}")
print(f"   Max Steps:       {grpo_config.max_steps}")
print(f"   Output Dir:      {grpo_config.output_dir}")
print(f"   Mixed Precision: bf16={grpo_config.bf16}, fp16={grpo_config.fp16}")

In [ ]:
# === GRPO-Trainer erstellen und Training starten ===
try:
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        args=grpo_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        reward_funcs=reward_func,
    )

    print("🚀 Starte GRPO-Training...")
    trainer.train()

    # Modell speichern
    output_dir = config.grpo.output_dir
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"✅ Training abgeschlossen! Modell gespeichert in: {output_dir}")

except NameError:
    print("⚠️  Training übersprungen — Modell wurde nicht geladen (keine GPU?).")
    print("   Das Notebook demonstriert den vollständigen Workflow.")
    print("   Für echtes Training: Führe das Notebook auf einer GPU-Maschine aus.")
except Exception as e:
    print(f"❌ Fehler beim Training: {e}")

## 7. Zusammenfassung

### GRPO-Training Workflow

```
┌─────────────┐    ┌──────────────┐    ┌───────────────┐    ┌──────────────┐
│ 1. Config   │ →  │ 2. Daten     │ →  │ 3. Modell     │ →  │ 4. Training  │
│  auswählen  │    │  laden       │    │  + LoRA laden │    │  mit GRPO    │
└─────────────┘    └──────────────┘    └───────────────┘    └──────────────┘
                                                                │
                                                    ┌───────────┘
                                                    ▼
                                            ┌──────────────┐
                                            │ 5. Modell    │
                                            │  speichern   │
                                            └──────────────┘
```

### Wichtige Parameter

| Parameter | Beschreibung | Typischer Wert |
|---|---|---|
| `num_generations` | Samples pro Prompt (Gruppengröße) | 4 |
| `beta` | KL-Divergence-Koeffizient | 0.04 |
| `learning_rate` | Lernrate | 5e-6 |
| `lora_r` | LoRA Rank | 16 |
| `lora_alpha` | LoRA Skalierung | 32 |
| `temperature` | Sampling-Temperatur | 0.9 |

### CLI-Aufruf

```bash
# Standard-Training (Qwen 7B)
python train_agent.py

# Llama 3.1 8B
python train_agent.py --model llama

# Schneller Test-Modus
python train_agent.py --test-mode

# Mit eigenen Daten
python train_agent.py --train-file data/train.jsonl --output-dir ./my_model
```

> **Repository:** [github.com/mark-baumann/ART](https://github.com/mark-baumann/ART)